# 🚗 Safespace Model Testing Notebook

This notebook allows you to test the accident detection model with:
- **Video files** - Test with pre-recorded footage
- **Camera feed** - Real-time detection from webcam
- **Single Images** - Static image testing

## 1. Setup

In [ ]:
# Cross-Platform Environment Setup
import os
import sys

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = 'KAGGLE_URL_BASE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print('Running in cloud environment. Installing dependencies...')
    !pip install -qU ultralytics wandb roboflow python-dotenv supervision
    
    if IN_COLAB:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
        os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
    elif IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY') or ''
        os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY') or ''
else:
    print('Running locally. Loading from .env...')
    try:
        from dotenv import load_dotenv
        load_dotenv('../.env')
    except ImportError:
        print('python-dotenv not installed. Please install it or set environment variables manually.')

In [ ]:
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import supervision as sv
from IPython.display import display, clear_output
import ipywidgets as widgets
import time

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Using device: {device}")

## 2. Load the Model

In [ ]:
# Configuration
MODEL_PATH = "../Models/accident_model.pt"  # Path to your model
CONFIDENCE_THRESHOLD = 0.5

if not Path(MODEL_PATH).exists():
    print(f"⚠️ Model not found at {MODEL_PATH}. Checking standard location...")
    MODEL_PATH = "../models/accident_classifier_and_detection.pt"

model = YOLO(MODEL_PATH)
model.to(device)
print(f"✅ Model loaded successfully from: {MODEL_PATH}")

## 3. Detection Helper Functions

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
def detect_and_annotate(frame, model, confidence=0.5):
    start_time = time.time()
    results = model.predict(frame, conf=confidence, device=device, verbose=False)
    inference_time = (time.time() - start_time) * 1000
    
    detections = sv.Detections.from_ultralytics(results[0])
    
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
    
    labels = [
        f"{model.names[class_id]} {conf:.2f}"
        for class_id, conf in zip(detections.class_id, detections.confidence)
    ] if len(detections) > 0 else []
    
    annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
    
    return annotated_frame, detections, inference_time

def add_stats_overlay(frame, fps, inference_time, detection_count):
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (250, 100), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    
    cv2.putText(frame, f"FPS: {fps:.1f}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(frame, f"Inference: {inference_time:.1f}ms", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(frame, f"Detections: {detection_count}", (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    return frame

def show_frame(window_name, frame):
    if IN_COLAB or IN_KAGGLE:
        _, encoded_img = cv2.imencode('.jpg', frame)
        display(widgets.Image(value=encoded_img.tobytes()))
        clear_output(wait=True)
    else:
        cv2.imshow(window_name, frame)

## 4. Test with Video File 🎬

In [ ]:
VIDEO_PATH = "../assets/Video Test/T1.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(f"❌ Error: Could not open video file: {VIDEO_PATH}")
else:
    print(f"✅ Video loaded: {VIDEO_PATH}")
    frame_count = 0
    fps_start_time = time.time()
    fps = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        annotated_frame, detections, inference_time = detect_and_annotate(frame, model, CONFIDENCE_THRESHOLD)
        
        if frame_count % 10 == 0:
            fps = 10 / (time.time() - fps_start_time)
            fps_start_time = time.time()
        
        annotated_frame = add_stats_overlay(annotated_frame, fps, inference_time, len(detections))
        
        if len(detections) > 0:
            cv2.putText(annotated_frame, "⚠️ ACCIDENT DETECTED!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
            
        show_frame("Safespace Model Test", annotated_frame)
        
        if not (IN_COLAB or IN_KAGGLE):
            if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    if not (IN_COLAB or IN_KAGGLE):
        cv2.destroyAllWindows()